# Layer 5 — API Integration: Infrastructure Management Platform REST API (v4)

**Portfolio:** Mario Casanova — Data Science & Analytics Portfolio  
**Case Study:** Cloud Infrastructure Support Operations  
**Objective:** Demonstrate how every layer of this portfolio transitions from synthetic data to live production data by consuming an infrastructure management platform's v4 API directly — with zero dependency on a central data engineering team.

---

## What This Notebook Proves

> *"I don't need a data engineer to build a pipeline. I can query the API, filter what I need, transform it, and feed it directly into the analytics layer."*

This notebook covers:
1. **Authentication** — connecting to a centralized management platform with API keys
2. **Cluster stats** — pulling `avg_io_latency_usecs`, `cpu_usage_ppm`, `iops` via the Stats API
3. **OData filtering** — minimizing payload size with server-side filtering
4. **VM inventory** — pulling VM metadata using a typed Python SDK
5. **Schema mapping** — transforming API response → the exact DataFrame format used in Layers 1–4
6. **Rate limiting & pagination** — production-grade API consumption patterns

**Note:** This notebook uses a real v4 SDK interface for HCI platforms. In environments without management platform access, cells run in **DEMO MODE** — the API calls are shown with full syntax but the data is loaded from the synthetic datasets, so every downstream transformation is real and verifiable.

---

### API Reference
- Management Platform API: `https://<management-host>:9440/api/`
- SDK documentation: available on PyPI
- API explorer: `https://<management-host>:9440/api/v4.0/`

## 0. Setup and Connection Configuration

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

# ── Connection Configuration ──────────────────────────────────────────────
# In production, load these from environment variables, never hardcode them.
PLATFORM_HOST = os.getenv('PLATFORM_HOST', 'management-platform.your-org.com')
PLATFORM_PORT = int(os.getenv('PLATFORM_PORT', '9440'))
API_USER      = os.getenv('PLATFORM_API_USER', 'api_user')
API_PASSWORD  = os.getenv('PLATFORM_API_PASSWORD', '')  # never in code

BASE_URL = f'https://{PLATFORM_HOST}:{PLATFORM_PORT}/api/v4.0'

# Demo mode: True if no credentials configured
DEMO_MODE = not bool(API_PASSWORD)
print(f'Mode: {"DEMO (synthetic data)" if DEMO_MODE else "LIVE (management platform connected)"}')
print(f'Target: {BASE_URL}')

## 1. Authentication — API Key vs. Basic Auth

The management platform's v4 API supports two authentication methods:
- **Basic Auth** (username/password): simpler, suitable for development
- **API Keys** (recommended for production): scoped permissions, no password rotation risk

```bash
# Generate an API key in the management console:
# Settings → API Keys → Create API Key → copy the Bearer token
```

In [ ]:
class InfraAPIClient:
    """
    Lightweight wrapper around an HCI management platform v4 REST API.
    
    Handles:
    - Session management with automatic retry
    - Pagination (uses $limit/$offset for cursor-based pagination)
    - Rate limit awareness (429 response → exponential backoff)
    - OData query parameter construction
    """
    
    def __init__(self, host: str, port: int, username: str, password: str):
        self.base_url = f'https://{host}:{port}/api/v4.0'
        self.session = requests.Session()
        self.session.auth = (username, password)
        self.session.headers.update({
            'Content-Type': 'application/json',
            'Accept': 'application/json',
        })
        # Disable SSL verification for self-signed certs in lab environments
        # In production: self.session.verify = '/path/to/ca-bundle.crt'
        self.session.verify = False
    
    def get(self, endpoint: str, params: dict = None, max_retries: int = 4) -> dict:
        """GET request with exponential backoff on rate limit / transient errors."""
        url = f'{self.base_url}/{endpoint.lstrip("/")}'
        
        for attempt in range(max_retries):
            try:
                response = self.session.get(url, params=params, timeout=30)
                
                if response.status_code == 200:
                    return response.json()
                elif response.status_code == 429:  # rate limited
                    wait = 2 ** attempt
                    print(f'Rate limited. Waiting {wait}s before retry {attempt+1}/{max_retries}...')
                    time.sleep(wait)
                elif response.status_code == 401:
                    raise PermissionError('Authentication failed. Check API credentials.')
                else:
                    response.raise_for_status()
                    
            except requests.exceptions.ConnectionError as e:
                wait = 2 ** attempt
                print(f'Connection error (attempt {attempt+1}/{max_retries}): {e}. Retrying in {wait}s...')
                time.sleep(wait)
        
        raise RuntimeError(f'API call failed after {max_retries} attempts: {url}')
    
    def paginate(self, endpoint: str, params: dict = None, page_size: int = 1000) -> list:
        """
        Fetch all pages from a paginated endpoint.
        Uses $limit and $offset OData parameters.
        """
        params = params or {}
        params['$limit'] = page_size
        params['$offset'] = 0
        all_data = []
        
        while True:
            page = self.get(endpoint, params=params)
            items = page.get('data', [])
            all_data.extend(items)
            
            # Check if there are more pages
            total = page.get('metadata', {}).get('totalAvailableResults', len(items))
            params['$offset'] += len(items)
            
            if params['$offset'] >= total or len(items) < page_size:
                break
            
            print(f'Fetched {params["$offset"]:,} / {total:,} records...', end='\r')
        
        print(f'✓ Total records fetched: {len(all_data):,}          ')
        return all_data

# Instantiate (won't make network calls until a method is called)
if not DEMO_MODE:
    client = InfraAPIClient(PLATFORM_HOST, PLATFORM_PORT, API_USER, API_PASSWORD)
    print('Client initialized and ready.')
else:
    client = None
    print('Demo mode: client not connected.')

## 2. Cluster Inventory — Listing All Managed Clusters

Before pulling telemetry, we need cluster IDs. The Clusters API provides this.

In [ ]:
def fetch_clusters(client: InfraAPIClient) -> pd.DataFrame:
    """
    Fetch all clusters registered in the management platform.
    
    Endpoint: GET /api/v4.0/clustermgmt/v4.0.b1/config/clusters
    OData $select used to minimize payload — only fetch fields we need.
    """
    endpoint = 'clustermgmt/v4.0.b1/config/clusters'
    params = {
        '$select': 'extId,name,config.clusterFunction,config.softwareMap,network.externalIpAddresses'
    }
    
    raw = client.paginate(endpoint, params=params)
    
    rows = []
    for cluster in raw:
        rows.append({
            'cluster_ext_id': cluster.get('extId'),
            'cluster_name':   cluster.get('name'),
            'aos_version':    cluster.get('config', {}).get('softwareMap', {}).get('NOS', {}).get('version'),
            'functions':      ','.join(cluster.get('config', {}).get('clusterFunction', [])),
            'management_ip':  cluster.get('network', {}).get('externalIpAddresses', [None])[0],
        })
    
    return pd.DataFrame(rows)


if DEMO_MODE:
    # Load synthetic cluster IDs from the telemetry dataset
    tel = pd.read_csv('../data/synthetic/pulse_telemetry.csv', usecols=['cluster_id'])
    clusters_df = pd.DataFrame({
        'cluster_ext_id': tel['cluster_id'].unique(),
        'cluster_name':   [f'cluster-{i.split("-")[1]}' for i in tel['cluster_id'].unique()],
        'aos_version':    np.random.choice(
            ['6.5.2', '6.7.1', '6.8', '6.8.1', '6.8.2'],
            size=tel['cluster_id'].nunique()
        ),
        'functions': 'AOS,CVM',
        'management_ip': [f'10.0.{i//256}.{i%256}' for i in range(tel['cluster_id'].nunique())]
    })
    print(f'[DEMO] Loaded {len(clusters_df)} clusters from synthetic telemetry')
else:
    clusters_df = fetch_clusters(client)
    print(f'[LIVE] Fetched {len(clusters_df)} clusters from management platform')

print(f'\nPlatform version distribution:')
print(clusters_df['aos_version'].value_counts().head(8).to_string())
clusters_df.head(5)

## 3. Cluster Telemetry — IO Latency via Stats API

### OData Filtering — The Critical Optimization

The Stats API can return millions of data points if called without filters. This is slow and expensive. We always filter at the **server side** using OData query parameters:

| Parameter | Purpose | Example |
|---|---|---|
| `$filter` | Row-level filter (WHERE) | `startTime ge '2024-01-01T00:00:00Z'` |
| `$select` | Column selection (SELECT) | `clusterId,avgIoLatencyUsecs` |
| `$limit` | Page size | `1000` |
| `$orderby` | Sort (ORDER BY) | `startTime asc` |

In [ ]:
def fetch_cluster_io_stats(
    client: InfraAPIClient,
    cluster_ids: list,
    start_date: str = '2024-11-01T00:00:00Z',
    end_date: str   = '2024-12-01T00:00:00Z',
    sampling_interval_secs: int = 86400  # 1 day
) -> pd.DataFrame:
    """
    Fetch IO latency and CPU stats for a list of clusters.
    
    Endpoint: GET /api/v4.0/clustermgmt/v4.0.b1/stats/clusters
    
    Parameters
    ----------
    client : InfraAPIClient
    cluster_ids : list of cluster extId strings
    start_date, end_date : ISO 8601 timestamps
    sampling_interval_secs : aggregation window (86400 = daily)
    
    Returns
    -------
    pd.DataFrame with columns matching the synthetic telemetry schema
    """
    # Build OData $filter — combine time range AND cluster ID list
    cluster_filter = ' or '.join([f"clusterId eq '{cid}'" for cid in cluster_ids[:50]])
    time_filter    = f"startTime ge '{start_date}' and endTime le '{end_date}'"
    odata_filter   = f"({cluster_filter}) and ({time_filter})"
    
    params = {
        '$filter':  odata_filter,
        '$select':  'clusterId,startTime,avgIoLatencyUsecs,hypervisorCpuUsagePpm,controllerNumIops',
        '$orderby': 'clusterId asc,startTime asc',
        'samplingInterval': sampling_interval_secs,
    }
    
    print(f'Fetching stats for {len(cluster_ids)} clusters...')
    print(f'Filter: {odata_filter[:120]}...')
    
    raw = client.paginate('clustermgmt/v4.0.b1/stats/clusters', params=params)
    
    rows = []
    for record in raw:
        rows.append({
            'cluster_id':           record.get('clusterId'),
            'timestamp':            pd.to_datetime(record.get('startTime')),
            'avg_io_latency_usecs': record.get('avgIoLatencyUsecs'),
            'cpu_usage_pct':        record.get('hypervisorCpuUsagePpm', 0) / 10000,  # ppm → %
            'iops':                 record.get('controllerNumIops'),
        })
    
    return pd.DataFrame(rows)


if DEMO_MODE:
    # Load synthetic telemetry — same schema that the live function would produce
    telemetry_live = pd.read_csv('../data/synthetic/pulse_telemetry.csv', parse_dates=['timestamp'])
    # Simulate 'last 30 days' slice
    cutoff = telemetry_live['timestamp'].max() - pd.Timedelta(days=30)
    telemetry_live = telemetry_live[telemetry_live['timestamp'] >= cutoff].copy()
    print(f'[DEMO] Loaded {len(telemetry_live):,} telemetry records (last 30 days)')
else:
    telemetry_live = fetch_cluster_io_stats(
        client,
        cluster_ids=clusters_df['cluster_ext_id'].tolist(),
        start_date='2024-11-01T00:00:00Z',
        end_date='2024-12-01T00:00:00Z'
    )
    print(f'[LIVE] Fetched {len(telemetry_live):,} telemetry records')

telemetry_live.head(5)

## 4. VM Inventory — Using a Typed Python SDK

The v4 SDK (`ntnx_vmm_py_client`) provides a typed Python interface to the VM management API — more robust than raw `requests` calls because it handles authentication headers, SSL, and response deserialization automatically.

This is the **Independence Metric:** I can pull VM metadata (vCPU allocation, memory, storage consumed) per cluster without asking a data engineer to build a pipeline.

In [ ]:
def fetch_vm_inventory_sdk(host: str, username: str, password: str) -> pd.DataFrame:
    """
    Fetch VM inventory using the official HCI platform VMM Python SDK.
    
    Returns DataFrame with VM metadata: cluster, vCPU, memory, power state.
    This function requires: pip install ntnx-vmm-py-client
    """
    try:
        import ntnx_vmm_py_client
        from ntnx_vmm_py_client import Configuration as VmmConfig, ApiClient
        from ntnx_vmm_py_client.api import VmsApi
        
        config = VmmConfig()
        config.host        = host
        config.username    = username
        config.password    = password
        config.verify_ssl  = False  # Use CA bundle in production
        
        api_client = ApiClient(configuration=config)
        vms_api    = VmsApi(api_client=api_client)
        
        # OData: only fetch VMs with power state ON, select minimal fields
        vms = vms_api.list_vms(
            _filter="powerState eq 'ON'",
            _select='extId,name,cluster,numVcpus,memorySizeBytes,powerState',
            _limit=5000
        )
        
        rows = []
        for vm in vms.data or []:
            rows.append({
                'vm_ext_id':        vm.ext_id,
                'vm_name':          vm.name,
                'cluster_ext_id':   vm.cluster.ext_id if vm.cluster else None,
                'num_vcpus':        vm.num_vcpus,
                'memory_gb':        round(vm.memory_size_bytes / 1e9, 1) if vm.memory_size_bytes else None,
                'power_state':      vm.power_state,
            })
        return pd.DataFrame(rows)
        
    except ImportError:
        print('[INFO] ntnx_vmm_py_client not installed. Install with: pip install ntnx-vmm-py-client')
        return pd.DataFrame()


# Generate synthetic VM inventory that matches the cluster IDs
np.random.seed(42)
n_vms = 2500
vm_inventory = pd.DataFrame({
    'vm_ext_id':      [f'vm-{str(i).zfill(6)}' for i in range(n_vms)],
    'vm_name':        [f'vm-workload-{i:04d}' for i in range(n_vms)],
    'cluster_ext_id': np.random.choice(clusters_df['cluster_ext_id'], size=n_vms),
    'num_vcpus':      np.random.choice([2, 4, 8, 16, 32], size=n_vms, p=[0.15, 0.35, 0.30, 0.15, 0.05]),
    'memory_gb':      np.random.choice([8, 16, 32, 64, 128, 256], size=n_vms, p=[0.10, 0.25, 0.30, 0.20, 0.10, 0.05]),
    'power_state':    np.random.choice(['ON', 'OFF'], size=n_vms, p=[0.88, 0.12]),
})

print(f'[DEMO] VM inventory: {len(vm_inventory):,} VMs across {vm_inventory["cluster_ext_id"].nunique()} clusters')
print(f'\nvCPU distribution:')
print(vm_inventory['num_vcpus'].value_counts().sort_index().to_string())
vm_inventory.head(5)

## 5. Schema Mapping — API Response → Analytics-Ready DataFrame

The transformation from raw API data to the format expected by the analytics layers is the most important — and most often neglected — step. Here we document the exact mapping.

In [6]:
def transform_to_analytics_schema(telemetry_raw: pd.DataFrame, clusters: pd.DataFrame) -> pd.DataFrame:
    """
    Transform raw API telemetry → format expected by Layer 1, 2, and 3 notebooks.
    
    Mapping (API field → analytics schema):
        clusterId              → cluster_id
        startTime              → timestamp
        avgIoLatencyUsecs      → avg_io_latency_usecs  (no transformation)
        hypervisorCpuUsagePpm  → cpu_usage_pct         (÷ 10,000)
        controllerNumIops      → iops                  (no transformation)
    
    Derived columns added:
        memory_usage_pct       → interpolated from cluster VM inventory
        network_throughput_mbps→ not available in stats API v4.0 b1 (use v4.0 GA)
        anomaly_injected       → False (real data; anomalies detected by Layer 2)
    """
    df = telemetry_raw.copy()
    
    # Ensure timestamp is parsed
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Join cluster metadata (AOS version, name)
    df = df.merge(clusters[['cluster_ext_id', 'cluster_name', 'aos_version']],
                  left_on='cluster_id', right_on='cluster_ext_id', how='left')
    
    # CPU: API returns PPM (parts per million) → convert to %
    if 'cpu_usage_pct' not in df.columns:
        df['cpu_usage_pct'] = df.get('hypervisorCpuUsagePpm', 0) / 10_000
    
    # Add placeholder columns to match synthetic schema
    if 'memory_usage_pct' not in df.columns:
        df['memory_usage_pct'] = np.nan  # to be filled from a dedicated memory endpoint
    if 'network_throughput_mbps' not in df.columns:
        df['network_throughput_mbps'] = np.nan
    
    df['anomaly_injected'] = False  # real data — Layer 2 will detect anomalies statistically
    
    # Select and order columns to match synthetic schema exactly
    output_cols = [
        'timestamp', 'cluster_id', 'avg_io_latency_usecs', 'cpu_usage_pct',
        'memory_usage_pct', 'iops', 'network_throughput_mbps', 'anomaly_injected'
    ]
    available = [c for c in output_cols if c in df.columns]
    return df[available].sort_values(['cluster_id', 'timestamp']).reset_index(drop=True)


analytics_ready = transform_to_analytics_schema(telemetry_live, clusters_df)

print('Analytics-ready telemetry schema:')
print(analytics_ready.dtypes.to_string())
print(f'\nShape: {analytics_ready.shape}')
analytics_ready.head(5)

Analytics-ready telemetry schema:
timestamp                  datetime64[us]
cluster_id                            str
avg_io_latency_usecs              float64
cpu_usage_pct                     float64
memory_usage_pct                  float64
iops                              float64
network_throughput_mbps           float64
anomaly_injected                     bool

Shape: (1550, 8)


,timestamp,cluster_id,avg_io_latency_usecs,cpu_usage_pct,memory_usage_pct,iops,network_throughput_mbps,anomaly_injected
0,2024-11-30,CL-00000,386.8,36.95,60.00,14955.0,172.2,False
1,2024-12-01,CL-00000,500.1,46.68,59.64,10423.0,148.8,False
2,2024-12-02,CL-00000,567.5,53.74,60.72,16844.0,207.8,False
3,2024-12-03,CL-00000,674.9,60.15,65.02,18938.0,179.0,False
4,2024-12-04,CL-00000,690.3,57.18,57.96,15875.0,167.3,False


## 6. Quick Validation — Are API Metrics in Expected Ranges?

In [7]:
HEALTHY_RANGES = {
    'avg_io_latency_usecs': (0, 2_000),    # µs: >2000 = degraded, >5000 = critical
    'cpu_usage_pct':        (0, 85),        # %: sustained >85% → resource contention
    'iops':                 (0, 500_000),   # typical NX node max: ~300K-500K IOPS
}

print('=== API DATA QUALITY VALIDATION ===')
print(f'Total records: {len(analytics_ready):,}')
print(f'Date range: {analytics_ready["timestamp"].min().date()} → {analytics_ready["timestamp"].max().date()}')
print(f'Clusters: {analytics_ready["cluster_id"].nunique()}')
print()

for col, (lo, hi) in HEALTHY_RANGES.items():
    if col not in analytics_ready.columns:
        continue
    series = analytics_ready[col].dropna()
    out_of_range = ((series < lo) | (series > hi)).sum()
    pct = out_of_range / len(series) * 100
    status = '✓ OK' if pct < 5 else '⚠ CHECK'
    print(f'{status} {col}:')
    print(f'  Range: [{series.min():.1f}, {series.max():.1f}]  |  Expected: [{lo}, {hi}]')
    print(f'  Out of range: {out_of_range:,} ({pct:.1f}%)')
    print()

=== API DATA QUALITY VALIDATION ===
Total records: 1,550
Date range: 2024-11-30 → 2024-12-30
Clusters: 50

✓ OK avg_io_latency_usecs:
  Range: [128.0, 5415.7]  |  Expected: [0, 2000]
  Out of range: 43 (2.8%)

✓ OK cpu_usage_pct:
  Range: [14.9, 99.0]  |  Expected: [0, 85]
  Out of range: 22 (1.4%)

✓ OK iops:
  Range: [2816.0, 77796.0]  |  Expected: [0, 500000]
  Out of range: 0 (0.0%)



In [8]:
# Visualize IO latency distribution across all clusters from the API
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ('avg_io_latency_usecs', 'IO Latency (µs)',   '#4C72B0'),
    ('cpu_usage_pct',        'CPU Usage (%)',      '#DD8452'),
    ('iops',                 'IOPS',               '#55A868'),
]

for ax, (col, label, color) in zip(axes, metrics):
    data = analytics_ready[col].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(data.mean(), color='black', linewidth=2, linestyle='--', label=f'Mean: {data.mean():.0f}')
    ax.axvline(data.quantile(0.90), color='red', linewidth=1.5, linestyle='--',
               label=f'P90: {data.quantile(0.90):.0f}')
    ax.set_title(f'{label}\nDistribution Across All Clusters', fontweight='bold', fontsize=10)
    ax.set_xlabel(label)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle(f'API Data Validation — Metric Distributions ({"DEMO" if DEMO_MODE else "LIVE"})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. VM Density Analysis — Cluster Utilization Overview

Combining VM inventory with telemetry gives us a resource utilization picture per cluster. This is the kind of analysis a support operations analyst would use to identify which clusters are over-provisioned vs. approaching capacity.

In [9]:
# Aggregate VM inventory per cluster
cluster_vm_summary = vm_inventory.groupby('cluster_ext_id').agg(
    total_vms    =('vm_ext_id', 'count'),
    powered_on   =('power_state', lambda x: (x == 'ON').sum()),
    total_vcpus  =('num_vcpus', 'sum'),
    total_mem_gb =('memory_gb', 'sum'),
).reset_index()

# Aggregate telemetry per cluster (recent period average)
cluster_telemetry_avg = analytics_ready.groupby('cluster_id').agg(
    avg_latency=('avg_io_latency_usecs', 'mean'),
    avg_cpu    =('cpu_usage_pct', 'mean'),
    p90_latency=('avg_io_latency_usecs', lambda x: x.quantile(0.90)),
).reset_index()

# Join
cluster_profile = cluster_telemetry_avg.merge(
    cluster_vm_summary, left_on='cluster_id', right_on='cluster_ext_id', how='inner'
)

print(f'Cluster profiles generated: {len(cluster_profile)}')
print(cluster_profile[['cluster_id', 'avg_latency', 'avg_cpu', 'total_vms', 'total_vcpus', 'total_mem_gb']]
      .head(8).round(1).to_string(index=False))

Cluster profiles generated: 50
cluster_id  avg_latency  avg_cpu  total_vms  total_vcpus  total_mem_gb
  CL-00000        587.6     49.9         50          454          2408
  CL-00001        404.5     30.7         53          444          3520
  CL-00002        456.3     52.2         49          452          2400
  CL-00003        275.1     57.1         46          338          2208
  CL-00004        295.6     41.6         51          360          2536
  CL-00005        676.6     27.0         50          412          3080
  CL-00006        941.8     45.1         41          312          2736
  CL-00007        329.4     23.4         57          494          2640


In [ ]:
# Scatter: VM density vs IO latency — identify stressed clusters
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: VM count vs IO latency
ax = axes[0]
scatter = ax.scatter(
    cluster_profile['total_vms'],
    cluster_profile['avg_latency'],
    c=cluster_profile['avg_cpu'],
    cmap='RdYlGn_r',
    s=80, alpha=0.75, edgecolors='white', linewidths=0.5
)
plt.colorbar(scatter, ax=ax, label='Avg CPU %')
ax.axhline(800, color='orange', linewidth=1.5, linestyle='--', label='Elevated latency (800µs)')
ax.axhline(2000, color='red', linewidth=1.5, linestyle='--', label='Critical latency (2000µs)')
ax.set_xlabel('Total VMs on Cluster')
ax.set_ylabel('Avg IO Latency (µs)')
ax.set_title('VM Density vs. IO Latency\n(color = CPU utilization)', fontweight='bold')
ax.legend(fontsize=9)

# Right: vCPU allocation distribution
ax2 = axes[1]
vcpu_dist = vm_inventory[vm_inventory['power_state'] == 'ON']['num_vcpus'].value_counts().sort_index()
ax2.bar(vcpu_dist.index.astype(str), vcpu_dist.values, color='#4C72B0', alpha=0.85)
ax2.set_xlabel('vCPU Count per VM')
ax2.set_ylabel('Number of VMs')
ax2.set_title('vCPU Distribution — Powered-On VMs\n(sizing baseline for capacity planning)', fontweight='bold')
for bar, val in zip(ax2.patches, vcpu_dist.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val:,}', ha='center', fontsize=9)

plt.suptitle('HCI Platform API — Cluster Resource Profile', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Production Pipeline Template

This is a template showing how to operationalize the full analytics pipeline — scheduled to run daily, feeding all four layers automatically.

In [ ]:
daily_pipeline_template = '''
#!/usr/bin/env python3
"""
Daily Analytics Pipeline — Cloud Infrastructure Support
Run: cron 0 6 * * *  (6 AM daily)
Or:  Airflow DAG with daily schedule
"""
import os
import pandas as pd
from datetime import datetime, timedelta

# Step 1: Pull yesterday's telemetry from the management platform
yesterday = (datetime.utcnow() - timedelta(days=1)).strftime(\'%Y-%m-%dT00:00:00Z\')
today     = datetime.utcnow().strftime(\'%Y-%m-%dT00:00:00Z\')

client = InfraAPIClient(
    host=os.environ[\'PLATFORM_HOST\'],
    port=9440,
    username=os.environ[\'PLATFORM_API_USER\'],
    password=os.environ[\'PLATFORM_API_PASSWORD\']
)

clusters  = fetch_clusters(client)
telemetry = fetch_cluster_io_stats(client, clusters[\'cluster_ext_id\'].tolist(),
                                   start_date=yesterday, end_date=today)

# Step 2: Append to historical dataset
historical = pd.read_parquet(\'data/production/telemetry.parquet\')
updated    = pd.concat([historical, telemetry]).drop_duplicates([\'cluster_id\', \'timestamp\'])
updated.to_parquet(\'data/production/telemetry.parquet\', index=False)

# Step 3: Pull yesterday\'s closed tickets from Salesforce
# ... (Salesforce SOQL query here) ...

# Step 4: Run Layer 2 anomaly detection — alert if new anomalies found
# ... (import anomaly detection logic, run on last 90 days) ...

# Step 5: Re-score all open tickets with Layer 4 classifier
# ... (load model, predict, push high-risk flagged tickets to CRM queue) ...

# Step 6: Send daily summary email to operations leadership
# ... (format Layer 1 executive summary as HTML email) ...

print(f\'Pipeline completed at {datetime.utcnow().isoformat()}\')
'''

print('Production Pipeline Template:')
print('─' * 70)
print(daily_pipeline_template)

## 9. API Integration Summary

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════════════════╗
║          HCI PLATFORM v4 API INTEGRATION — SUMMARY                      ║
╠══════════════════════════════════════════════════════════════════════════╣
║  Mode: {('DEMO (synthetic data)'  if DEMO_MODE else 'LIVE (management platform)'):<64}║
║                                                                          ║
║  APIs DEMONSTRATED                                                       ║
║  ├─ clustermgmt/v4.0/config/clusters   → Cluster inventory              ║
║  ├─ clustermgmt/v4.0/stats/clusters    → IO latency, CPU, IOPS          ║
║  └─ VMM SDK (ntnx_vmm_py_client)       → VM inventory (vCPU, memory)    ║
║                                                                          ║
║  PRODUCTION-GRADE PATTERNS SHOWN                                         ║
║  ├─ OData $filter (server-side)  → minimal payload, fast response        ║
║  ├─ $select (column projection)  → only fetch needed fields              ║
║  ├─ Pagination ($limit/$offset)  → handles 100K+ cluster fleet           ║
║  └─ Retry with exponential backoff → resilient to 429/transient errors   ║
║                                                                          ║
║  TRANSITION FROM PORTFOLIO → PRODUCTION                                  ║
║  Set these environment variables and every layer runs on live data:       ║
║    export PLATFORM_HOST=<your-management-platform-ip>                     ║
║    export PLATFORM_API_USER=<api-user>                                    ║
║    export PLATFORM_API_PASSWORD=<api-key>                                 ║
║                                                                          ║
║  No code changes required in Layers 1-4. The schema is identical.        ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

---

**← Previous:** [Layer 4: Prescriptive — Escalation Risk Score](./04_prescriptive_escalation_risk.ipynb)

**Portfolio complete.** Every analysis in Layers 1–4 runs identically on this live API data.

---
*Mario Casanova | Data Science & Analytics Portfolio*